# Visualize a Rollout

This notebook loads a saved rollout from `rollouts/episode_*.npz` and visualizes observations, rewards, and actions.

In [2]:
# Import Required Libraries
import os
import shutil
import subprocess
import tempfile
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Video

from env import CarRacingWrapper

plt.rcParams['figure.figsize'] = (8, 6)
plt.rcParams['image.interpolation'] = 'nearest'


In [3]:
# Collect Rollout Data

rollouts_dir = Path('rollouts')

# Replace with the episode file you want to visualize.
rollout_path = rollouts_dir / 'episode_006001.npz'
if not rollout_path.exists():
    raise FileNotFoundError(f"Rollout file not found: {rollout_path}")

with np.load(rollout_path) as rollout_data:
    obs = rollout_data['obs']
    actions = rollout_data['actions']
    rewards = rollout_data['rewards']
    terminated = rollout_data['terminated']
    truncated = rollout_data['truncated']
    seed = int(rollout_data['seed'][0])

print(f"Loaded rollout: {rollout_path}")
print(f"obs shape: {obs.shape}")
print(f"actions shape: {actions.shape}")
print(f"rewards shape: {rewards.shape}")
print(f"terminated shape: {terminated.shape}")
print(f"truncated shape: {truncated.shape}")
print(f"seed: {seed}")

Loaded rollout: rollouts/episode_006001.npz
obs shape: (0,)
actions shape: (1000, 3)
rewards shape: (1000,)
terminated shape: (1000,)
truncated shape: (1000,)
seed: 191664963


In [5]:
# Animate Rollout Sequence with ffmpeg

output_video_dir = Path('rollout_videos')
output_video_dir.mkdir(parents=True, exist_ok=True)

with tempfile.TemporaryDirectory() as tmpdir:
    tmpdir = Path(tmpdir)

    env = CarRacingWrapper(full_episode=True, render_mode='rgb_array')
    try:
        frame, _ = env.reset(seed=seed)
        frame_path = tmpdir / f'frame_{0:06d}.png'
        plt.imsave(frame_path, frame)

        for idx, action in enumerate(actions, start=1):
            frame, reward, terminated, truncated, info = env.step(action)
            frame_path = tmpdir / f'frame_{idx:06d}.png'
            plt.imsave(frame_path, frame)
    finally:
        env.close()

    video_path = output_video_dir / f'rollout_{rollout_path.stem}.mp4'
    ffmpeg_cmd = [
        'ffmpeg',
        '-y',
        '-framerate', '30',
        '-i', str(tmpdir / 'frame_%06d.png'),
        '-c:v', 'libx264',
        '-pix_fmt', 'yuv420p',
        str(video_path),
    ]

    try:
        subprocess.run(ffmpeg_cmd, check=True, capture_output=True)
    except FileNotFoundError:
        raise RuntimeError('ffmpeg is not installed or not found in PATH.')
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(f'ffmpeg failed: {exc.stderr.decode()}')

print(f'Video saved to: {video_path}')
Video(str(video_path), embed=True, html_attributes='controls')


Video saved to: rollout_videos/rollout_episode_006001.mp4
